# Notebook 3: `z` Motif Discovery and Analysis

This notebook discovers motif flows directly in `z`, checkpoints the clean motif artifacts once, and compares the frozen and joint branches on motif quality and structure.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")

if REPO_DIR.parent.exists() and "google.colab" in str(get_ipython()):
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import json
import torch

from flow_circuits.data import build_cifar10_splits
from flow_circuits.evaluation import (
    discover_motif_families,
    run_motif_gallery_experiment,
    run_motif_persistence_experiment,
    run_motif_topology_experiment,
)
from flow_circuits.training import load_components_from_checkpoint, load_yaml_config


## Parameters


In [ ]:
CONFIG_NAME = "resnet18_z_workflow"
CONFIG_PATH = Path("configs/flow/resnet18_aligned.yaml")
DISCOVERY_MAX_IMAGES = 5000
BOOTSTRAP_ITERATIONS = 5
MIN_MOTIF_LAYERS = 2
MIN_CLUSTER_SIZE = 20
MOTIF_OVERLAP_THRESHOLD = 0.50
TOP_MOTIFS_TO_RENDER = 8
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits") if Path("/content/drive/MyDrive").exists() else Path("artifacts/flow_circuits")
NB02_OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb02_q_validation" / CONFIG_NAME
SELECTION_ARTIFACT = NB02_OUTPUT_DIR / "selected_checkpoints.json"
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb03_z_motif_discovery_and_analysis" / CONFIG_NAME


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base_config = load_yaml_config(CONFIG_PATH)
selection = json.loads(SELECTION_ARTIFACT.read_text(encoding="utf-8"))
FROZEN_SELECTED_CHECKPOINT = Path(selection["frozen"]["checkpoint_path"])
JOINT_SELECTED_CHECKPOINT = Path(selection["joint"]["checkpoint_path"])
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=base_config["data"]["batch_size"],
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def _run_or_load(path: Path, fn):
    if path.exists() and not FORCE_RERUN:
        return _load_json(path)
    result = fn(path)
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result

RESULTS = {}
for tag, checkpoint_path in {"frozen": FROZEN_SELECTED_CHECKPOINT, "joint": JOINT_SELECTED_CHECKPOINT}.items():
    branch_dir = OUTPUT_DIR / tag
    branch_dir.mkdir(parents=True, exist_ok=True)
    components = load_components_from_checkpoint(checkpoint_path, device)
    motif_artifact = _run_or_load(
        branch_dir / f"{tag}_clean_motifs.json",
        lambda cache_path, tag=tag, components=components, checkpoint_path=checkpoint_path: discover_motif_families(
            components,
            loaders["discovery"],
            device=device,
            checkpoint_tag=tag,
            max_images=DISCOVERY_MAX_IMAGES,
            nodes_per_layer=components.config["tokenization"].get("grid_size", 4) ** 2,
            bootstrap_iterations=BOOTSTRAP_ITERATIONS,
            merge_threshold=MOTIF_OVERLAP_THRESHOLD,
            min_cluster_size=MIN_CLUSTER_SIZE,
            use_all_nodes=True,
            checkpoint_path=str(checkpoint_path),
            output_path=cache_path,
        ),
    )
    gallery = _run_or_load(
        branch_dir / "motif_gallery.json",
        lambda cache_path, tag=tag, components=components: run_motif_gallery_experiment(
            components,
            loaders["discovery"],
            device=device,
            checkpoint_tag=tag,
            motif_artifact=motif_artifact,
            max_images=DISCOVERY_MAX_IMAGES,
            topk=TOP_MOTIFS_TO_RENDER,
            output_path=cache_path,
        ),
    )
    persistence = _run_or_load(
        branch_dir / "motif_persistence.json",
        lambda cache_path, tag=tag: run_motif_persistence_experiment(
            motif_artifact,
            checkpoint_tag=tag,
            output_path=cache_path,
        ),
    )
    topology = _run_or_load(
        branch_dir / "motif_topology.json",
        lambda cache_path, tag=tag: run_motif_topology_experiment(
            motif_artifact,
            checkpoint_tag=tag,
            output_path=cache_path,
        ),
    )
    RESULTS[tag] = {
        "motif_artifact": motif_artifact,
        "motif_gallery": gallery,
        "motif_persistence": persistence,
        "motif_topology": topology,
    }

summary = {
    "frozen": {
        "n_motifs": RESULTS["frozen"]["motif_artifact"]["summary"]["n_motifs"],
        "mean_supporting_layers": RESULTS["frozen"]["motif_artifact"]["summary"]["mean_supporting_layers"],
        "mean_depth_span": RESULTS["frozen"]["motif_persistence"]["summary"]["mean_depth_span"],
        "mean_class_purity": RESULTS["frozen"]["motif_gallery"]["summary"]["mean_class_purity"],
    },
    "joint": {
        "n_motifs": RESULTS["joint"]["motif_artifact"]["summary"]["n_motifs"],
        "mean_supporting_layers": RESULTS["joint"]["motif_artifact"]["summary"]["mean_supporting_layers"],
        "mean_depth_span": RESULTS["joint"]["motif_persistence"]["summary"]["mean_depth_span"],
        "mean_class_purity": RESULTS["joint"]["motif_gallery"]["summary"]["mean_class_purity"],
    },
}
(OUTPUT_DIR / "motif_analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")


In [ ]:
for tag in ("frozen", "joint"):
    artifact = RESULTS[tag]["motif_artifact"]
    persistence = RESULTS[tag]["motif_persistence"]
    gallery = RESULTS[tag]["motif_gallery"]
    print(f"[{tag}] n_motifs={artifact['summary']['n_motifs']} | mean_layers={artifact['summary']['mean_supporting_layers']:.3f} | mean_span={persistence['summary']['mean_depth_span']:.3f} | purity={gallery['summary']['mean_class_purity']:.3f}")
    topologies = RESULTS[tag]["motif_topology"]["summary"]["topology_counts"]
    print(f"  topology_counts={topologies}")
    print()
